**Author:**

Kumar Jayant

[RFM-B²: RFM for Trade Partners, and Why Your Biggest Partner Might not be The “Best One”](https://aimldl.io/2026/11/01/rfm-b2-rfm-for-trade-partners/)

# RFM-B²: RFM for Trade Partners, and Why Your Biggest Partner Might not be The "Best One"

Companion notebook to the aimldl.io post (Part 2: trade partners — dealers, distributors and resellers).

**Data:** Microsoft AdventureWorks reseller sales (MIT License), a fictional bike manufacturer selling through 701 resellers. Three tables (FactResellerSales, DimReseller, DimPromotion) merged into one clean file on Kaggle: `krjynt/rfm-for-b2b-reseller-sales-with-schemes. The data is used as an illustration of the framework, not as evidence.

**What this notebook does**
1. Prepares the data (snapshot date and rolling 12-month window, net revenue after scheme discounts, dealers who never ordered set aside as a prospects list).
2. Computes the six measures (Recency, Frequency, Monetary, Wallet share, Absolute gross margin, % gross margin) and scores each 1–5.
3. Reads the scores through three lenses (Activity, Scale & potential, Profitability), calls each lens High, Mid or Low, and places every dealer in one of 27 cells.
4. Maps the 27 cells to nine personas, each with an action on promotion spend, visits and PJP, and incentives.
5. Profiles the personas and draws the charts.

> **Settings to tune for your own business** (cell 2): the snapshot date and window, the scoring bands, and the lens cut-offs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import KBinsDiscretizer
import glob, os
import kagglehub
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import FancyBboxPatch
import matplotlib.ticker as mticker

pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Settings

Change parameters here

* **Snapshot date:** 30 Nov 2013, the day after the last order in the data, so Recency is never 0.
* **Window:** exactly 12 months ending on the snapshot date (1 Dec 2012 – 30 Nov 2013). Partners in this data order on month-ends, so the window starts on 1 Dec to avoid counting one extra quarter's orders. For your own data, use a rolling 12 months, or align the window to your incentive year.
* **Ordering cycle:** one quarter (about 92 days). Recency and Frequency bands are set against it.
* **Recency (days since last order):** 5 = up to 30 · 4 = 31–60 · 3 = 61–92 (still within one cycle) · 2 = 93–183 (missed one cycle) · 1 = more than 183 (missed two or more).
* **Frequency (orders in the window):** 5 = 5 or more · 4 = 4 (every quarter) · 3 = 3 · 2 = 1–2 · 1 = none.
* **Monetary (net revenue after scheme discounts):** five bands found by KMeans on log(1 + revenue) of partners who ordered in the window. No scaling needed for a single measure.
* **Wallet share (our revenue ÷ the partner's annual sales, capped at 100%):** 5 = above 25% · 4 = above 10% up to 25% · 3 = above 5% up to 10% · 2 = above 1% up to 5% · 1 = 1% or less.
* **Absolute gross margin (net revenue − cost):** 5 = above 5,000 · 4 = above 1,000 up to 5,000 · 3 = above 0 up to 1,000 · 2 = above −5,000 up to 0 · 1 = −5,000 or less.
* **% gross margin (gross margin ÷ net revenue):** 5 = above 10% · 4 = above 5% up to 10% · 3 = above 0% up to 5% · 2 = above −5% up to 0% · 1 = −5% or less.
* **No orders in the window:** score 1 on Frequency, Monetary, Wallet share and both margins. Recency is still scored.
* **Never ordered:** set aside as the **Never Started** prospects list, outside the grid.
* **Lens cut-offs** (lens score = average of its two measures): Low = 2.0 or below · Mid = above 2.0 up to 3.5 · High = above 3.5.

> These bands and cut-offs are a starting point. Tune them to your ordering cycle, margins and objectives.

In [ ]:
# Data: pulled straight from Kaggle (works in Colab and in Kaggle notebooks; cached after the first download).
# To run on your own file instead, comment out the next two lines and set DATA_PATH to your CSV, e.g.
# DATA_PATH = '/content/rfmb2b.csv'

path = kagglehub.dataset_download('krjynt/rfm-for-b2b-reseller-sales-with-schemes')
DATA_PATH = glob.glob(os.path.join(path, '*.csv'))[0]
print(DATA_PATH)

In [ ]:
# DATA_PATH = 'rfmb2b.csv'   # Colab: upload the file; Kaggle: '/kaggle/input/rfm-for-b2b-reseller-sales-with-schemes/rfmb2b.csv'

WINDOW_START = pd.Timestamp('2012-12-01')
WINDOW_END   = pd.Timestamp('2013-11-30')            # exclusive: data runs to 29 Nov 2013
SNAPSHOT     = WINDOW_END                            # 'today' for Recency = last day + 1

# Ordering cycle: partners order about once a quarter
CYCLE_DAYS = 92

# Scoring bands, used with pd.cut(..., right=True): each band runs "above the lower edge, up to the upper edge"
RECENCY_BINS    = [0, 30, 60, 92, 183, float('inf')]       # days since last order
RECENCY_LABELS  = [5, 4, 3, 2, 1]                          # recent = 5

FREQUENCY_BINS   = [-1, 0, 2, 3, 4, float('inf')]          # orders in the window
FREQUENCY_LABELS = [1, 2, 3, 4, 5]                         # 0 = 1, 1-2 = 2, 3 = 3, 4 = 4 (every quarter), 5+ = 5

M_BANDS = 5                                                # Monetary: KMeans on log(1 + net revenue), no scaling needed

WALLET_BINS   = [-1, 0.01, 0.05, 0.10, 0.25, float('inf')] # revenue ÷ AnnualSales, capped at 1.0 before scoring
WALLET_LABELS = [1, 2, 3, 4, 5]

GM_BINS   = [-float('inf'), -5000, 0, 1000, 5000, float('inf')]   # absolute gross margin, USD
GM_LABELS = [1, 2, 3, 4, 5]

GMP_BINS   = [-float('inf'), -0.05, 0, 0.05, 0.10, float('inf')]  # % gross margin = gross margin ÷ net revenue
GMP_LABELS = [1, 2, 3, 4, 5]

# Lens cut-offs: lens score = average of its two measures
LENS_BINS   = [0, 2.0, 3.5, 5]                             # Low <= 2.0 < Mid <= 3.5 < High
LENS_LABELS = ['L', 'M', 'H']

## 2. Load the data

In [ ]:
raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig', dtype={'SalesOrderNumber': str})
raw['OrderDate'] = pd.to_datetime(raw['OrderDate'], format='%Y-%m-%d')

orders   = raw[raw['SalesOrderNumber'].notna()]      # order lines
never    = raw[raw['SalesOrderNumber'].isna()]       # dealers who never ordered (one row each)

print(f'{len(raw):,} rows | {orders.OrderDate.min():%d %b %Y} to {orders.OrderDate.max():%d %b %Y}')
print(f'{raw.ResellerKey.nunique():,} dealers | {orders.ResellerKey.nunique():,} ordered | {len(never):,} never ordered')
raw.head()

## 3. Prepare the data

Coding checklist, in order:

1. Dates are already converted above.
2. Set aside the dealers who never ordered: they become the **Never Started** prospects list and sit outside the grid.
3. Check the order lines: no duplicate lines (`SalesOrderNumber` + `SalesOrderLineNumber`), and every line has a dealer (`ResellerKey`).
4. Keep only the 12-month window (1 Dec 2012 – 29 Nov 2013) for Frequency, Monetary, Wallet share and both margins. Recency uses each dealer's last order ever, so dealers who were quiet all year stay in the base and score 1 on everything else.
5. Use **net revenue** = `SalesAmount` (already after scheme discounts) and **gross margin** = `Margin` (`SalesAmount − TotalProductCost`).
6. Keep one row per dealer for its profile (`ResellerName`, `BusinessType`, `AnnualSales`), which is needed for Wallet share.

> **On your own data**, add two steps before step 4: net off returns and credit notes, and drop non-product lines (freight, adjustments, charges). This sample has neither.

In [ ]:
steps = [('Raw rows', len(raw))]

# Never Started: dealers with no orders at all -> prospects list, outside the grid
never_started = never[['ResellerKey', 'ResellerName', 'BusinessType', 'AnnualSales']].copy()
steps.append(('Never-ordered dealers set aside', len(never_started)))

df = orders.copy()
steps.append(('Order lines', len(df)))

# Checks: no duplicate lines, every line has a dealer
dupes = df.duplicated(['SalesOrderNumber', 'SalesOrderLineNumber']).sum()
no_dealer = df['ResellerKey'].isna().sum()
print(f'Duplicate order lines: {dupes:,} | Lines with no dealer: {no_dealer:,}')
df = df.drop_duplicates(['SalesOrderNumber', 'SalesOrderLineNumber']).dropna(subset=['ResellerKey'])
steps.append(('After checks', len(df)))

# Full history is kept for Recency (last order ever)
history = df

# 12-month window for Frequency, Monetary, Wallet share and both margins
window = df[(df.OrderDate >= WINDOW_START) & (df.OrderDate < WINDOW_END)]
steps.append(('In the 12-month window', len(window)))

# One row per dealer: profile needed for Wallet share
profile = (history.groupby('ResellerKey')[['ResellerName', 'BusinessType', 'AnnualSales']]
                  .first())

print(f'Window: {window.SalesOrderNumber.nunique():,} orders from {window.ResellerKey.nunique():,} dealers | '
      f'net revenue {window.SalesAmount.sum():,.0f} | gross margin {window.Margin.sum():,.0f}')
pd.DataFrame(steps, columns=['Step', 'Rows'])

## 4. Compute the six measures

* **Recency:** days from the dealer's last order (ever) to the snapshot date (30 Nov 2013). Never 0: a last-day buyer scores 1 day.
* **Frequency:** distinct orders (`SalesOrderNumber`) in the 12-month window, not order lines.
* **Monetary:** net revenue in the window = sum of `SalesAmount` (already after scheme discounts).
* **Wallet share:** Monetary ÷ the dealer's `AnnualSales`, capped at 100% (annual sales come in bands, so a few dealers would otherwise go above 100%).
* **Absolute gross margin:** sum of `Margin` in the window (net revenue − product cost).
* **% gross margin:** absolute gross margin ÷ net revenue.

Dealers with no orders in the window keep their Recency; Frequency is 0, and the other four measures are left blank and scored 1 in the next step.

In [ ]:
def compute_measures(history, window, profile, snapshot):
    m = profile.copy()                                                    # one row per dealer who ever ordered

    # Recency: last order ever -> snapshot (never 0)
    m['Recency'] = (snapshot - history.groupby('ResellerKey')['OrderDate'].max()).dt.days

    # Frequency, Monetary, gross margin: 12-month window only
    w = (window.groupby('ResellerKey')
               .agg(Frequency=('SalesOrderNumber', 'nunique'),             # distinct orders, not lines
                    Monetary=('SalesAmount', 'sum'),                       # net revenue after schemes
                    GrossMargin=('Margin', 'sum')))                        # net revenue - cost
    m = m.join(w)
    m['Frequency'] = m['Frequency'].fillna(0).astype(int)                 # quiet all year = 0 orders

    m['WalletShare']    = (m['Monetary'] / m['AnnualSales']).clip(upper=1.0)  # capped at 100%
    m['GrossMarginPct'] = m['GrossMargin'] / m['Monetary']
    return m

measures = compute_measures(history, window, profile, SNAPSHOT)
active = measures['Frequency'] > 0
print(f'{len(measures):,} dealers | {active.sum():,} ordered in the window | {(~active).sum():,} quiet all year')
print(f'Net revenue {measures.Monetary.sum():,.0f} | gross margin {measures.GrossMargin.sum():,.0f}')
measures[['Recency', 'Frequency', 'Monetary', 'WalletShare', 'GrossMargin', 'GrossMarginPct']].describe(percentiles=[.25, .5, .75, .9])

## 5. Score each 1–5

* **Recency:** cut-offs set against the quarterly ordering cycle: up to 30 days = 5, 31–60 = 4, 61–92 = 3 (still within one cycle), 93–183 = 2 (missed one cycle), more than 183 = 1 (missed two or more).
* **Frequency:** fixed order counts in the window: 0 = 1, 1–2 = 2, 3 = 3, 4 (every quarter) = 4, 5 or more = 5. (Quintiles fail here: most dealers have exactly 4 orders.)
* **Monetary:** five natural bands found by 1-D KMeans on log(1 + net revenue) of dealers who ordered in the window. No scaling needed for a single measure.
* **Wallet share:** above 25% = 5, above 10% up to 25% = 4, above 5% up to 10% = 3, above 1% up to 5% = 2, 1% or less = 1.
* **Absolute gross margin:** above 5,000 = 5, above 1,000 up to 5,000 = 4, above 0 up to 1,000 = 3, above −5,000 up to 0 = 2, −5,000 or less = 1.
* **% gross margin:** above 10% = 5, above 5% up to 10% = 4, above 0% up to 5% = 3, above −5% up to 0% = 2, −5% or less = 1.
* **Dealers with no orders in the window** score 1 on Monetary, Wallet share and both margins (Frequency already scores 1).

> **Keep the Monetary bands fixed between runs.** KMeans finds new bands every time you re-run, so a dealer's Monetary score can change next quarter even when their revenue didn't. Fit the bands once and reuse them with `kb.transform(...)`, so quarter-on-quarter persona moves reflect the dealer, not the bands.

In [ ]:
def score_measures(measures, kb=None):
    out = measures.copy()
    active = out['Frequency'] > 0                                     # ordered in the window

    out['R'] = pd.cut(out['Recency'],   bins=RECENCY_BINS,   labels=RECENCY_LABELS).astype(int)
    out['F'] = pd.cut(out['Frequency'], bins=FREQUENCY_BINS, labels=FREQUENCY_LABELS).astype(int)

    # Monetary: KMeans bands on log revenue, fitted on active dealers only
    log_m = np.log1p(out.loc[active, ['Monetary']].clip(lower=0))
    if kb is None:                                                    # first run: fit the bands
        kb = KBinsDiscretizer(n_bins=M_BANDS, encode='ordinal', strategy='kmeans', subsample=None)
        kb.fit(log_m)
    out['M'] = 1
    out.loc[active, 'M'] = kb.transform(log_m).ravel().astype(int) + 1

    # Wallet share and both margins: fixed business bands; no orders in the window = 1
    for col, src, bins, labels in [('W',   'WalletShare',    WALLET_BINS, WALLET_LABELS),
                                   ('GM',  'GrossMargin',    GM_BINS,     GM_LABELS),
                                   ('GMP', 'GrossMarginPct', GMP_BINS,    GMP_LABELS)]:
        out[col] = pd.cut(out[src], bins=bins, labels=labels).astype(float).fillna(1).astype(int)
    return out, kb

scored, kb = score_measures(measures)
m_edges = np.expm1(kb.bin_edges_[0])
print('M band edges (USD):', ' | '.join(f'${e:,.0f}' for e in m_edges))
for col in ['R', 'F', 'M', 'W', 'GM', 'GMP']:
    print(col, scored[col].value_counts().sort_index().to_dict())

## 6. Reading the score through lenses

**What the lenses show on this sample (illustration only)**

* **Activity:** 333 dealers are High, 123 Mid and 179 Low. The Low group is mostly the 142 dealers who placed no orders in the last 12 months.
* **Scale & potential:** only 155 dealers are High. Most dealers (297) are Low, either because they buy little from us or because we are a small share of their business.
* **Profitability:** 439 dealers are Low, against 146 High. This sample's margins are unusually thin, so read this as how the lens works, not as a benchmark.
* **Cells:** dealers occupy 19 of the 27 cells. The largest are L/L/L (157 dealers gone quiet), H/M/L (94 active, mid-scale, low-profit), H/H/L (81 active, big, low-profit), H/M/H (45) and H/H/H (36).

The same revenue can sit in very different cells. H/H/L and H/H/H are both active and big, but one earns the business very little. That's why each cell gets its own persona and action next, instead of one composite score.

In [ ]:
def build_lenses(scored):
    out = scored.copy()
    out['Activity_score']      = (out['R']  + out['F'])   / 2
    out['Scale_score']         = (out['M']  + out['W'])   / 2
    out['Profitability_score'] = (out['GM'] + out['GMP']) / 2

    for lens in ['Activity', 'Scale', 'Profitability']:
        out[lens] = pd.cut(out[f'{lens}_score'], bins=LENS_BINS, labels=LENS_LABELS).astype(str)

    out['Cell'] = out['Activity'] + '/' + out['Scale'] + '/' + out['Profitability']
    return out

lenses = build_lenses(scored)
for lens in ['Activity', 'Scale', 'Profitability']:
    print(lens, lenses[lens].value_counts().reindex(['H', 'M', 'L']).to_dict())
print(f"{lenses['Cell'].nunique()} of 27 cells occupied")
lenses['Cell'].value_counts().head(10)

In [ ]:
PERSONA_COLOURS = {                       # (fill, text) — same as the blog graphics
    'Business Sustainers': ('#d6e6fb', '#174ea6'), 'Ringfence Them':     ('#e6dcfa', '#5b2fb0'),
    'Volume Movers':       ('#fde0c2', '#9a4a00'), 'On the Fence':       ('#d3eedd', '#17663a'),
    'Business Bleeders':   ('#f8cfcc', '#a3160d'), 'Potential Goldmine': ('#f8ebb8', '#7a5a00'),
    'Costly Tail':         ('#e4e6e9', '#3d434a'), 'Gone Quiet':         ('#c9ccd1', '#15181c')}
LENS_COLOURS = {'Activity': '#2f7bd9', 'Scale': '#1f8a5b', 'Profit': '#c2610c'}

def persona_of(a, s, p):
    if a == 'L': return 'Gone Quiet'
    if s == 'H': return ('Business Sustainers' if a == 'H' else 'Ringfence Them') if p == 'H' else 'Volume Movers'
    if s == 'M': return 'On the Fence' if p == 'H' else 'Business Bleeders'
    return 'Potential Goldmine' if p in 'HM' else 'Costly Tail'

# Inputs for the chart: built here so the cell runs on its own
levels = ['H', 'M', 'L']
names  = {'H': 'High', 'M': 'Mid', 'L': 'Low'}

nine_box = {}
for a in levels:
    sub = lenses[lenses['Activity'] == a]
    nine_box[a] = (pd.crosstab(sub['Scale'], sub['Profitability'])
                     .reindex(index=levels, columns=levels, fill_value=0))

fig, axes = plt.subplots(1, 3, figsize=(18, 6.8))
fig.patch.set_facecolor('#f9f8f6')

for ax, a in zip(axes, levels):
    grid = nine_box[a]
    ax.set_facecolor('#f9f8f6')
    for i, s in enumerate(levels):
        for j, p in enumerate(levels):
            n = int(grid.iloc[i, j])
            per = persona_of(a, s, p)
            fill, ink = PERSONA_COLOURS[per]
            ink_shown = ink if n else '#b0b4ba'                       # empty cells greyed out
            ax.add_patch(FancyBboxPatch((j + 0.05, 2 - i + 0.05), 0.9, 0.9,
                                        boxstyle='round,pad=0,rounding_size=0.06',
                                        facecolor=fill if n else '#ffffff', edgecolor=fill, linewidth=2))
            ax.text(j + 0.5, 2 - i + 0.58, f'{n}', ha='center', va='center', fontsize=24, fontweight='bold', color=ink_shown)
            ax.text(j + 0.5, 2 - i + 0.25, per, ha='center', va='center', fontsize=9.5, color=ink_shown)
            ax.text(j + 0.08, 2 - i + 0.86, f'{a}/{s}/{p}', ha='left', va='center', fontsize=8, color=ink, alpha=0.7)

    ax.set_xlim(0, 3); ax.set_ylim(0, 3)
    ax.set_xticks([0.5, 1.5, 2.5]); ax.set_xticklabels(['High', 'Mid', 'Low'], fontsize=11, fontweight='bold', color=LENS_COLOURS['Profit'])
    ax.set_yticks([2.5, 1.5, 0.5]); ax.set_yticklabels(['High', 'Mid', 'Low'], fontsize=11, fontweight='bold', color=LENS_COLOURS['Scale'])
    ax.xaxis.tick_top(); ax.tick_params(length=0, pad=4)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_xlabel('Profitability →', fontsize=11, color=LENS_COLOURS['Profit'], labelpad=10)
    ax.xaxis.set_label_position('top')
    ax.set_ylabel('Scale & potential', fontsize=11, color=LENS_COLOURS['Scale'], labelpad=8)
    ax.set_title(f'Activity: {names[a]}  ·  {int(grid.values.sum()):,} partners',
                 fontsize=15, fontweight='bold', color=LENS_COLOURS['Activity'], pad=42)

fig.suptitle('Where the trade partners sit: 27 cells, coloured by persona', fontsize=19, fontweight='bold', x=0.02, ha='left', y=1.02)
fig.text(0.02, -0.02, f'{len(lenses):,} partners who ever ordered. The {len(never_started):,} who never ordered sit outside the grid '
         '(Never Started). Illustration on the AdventureWorks sample.', fontsize=10, color='#6b7178')
plt.tight_layout()

os.makedirs('charts', exist_ok=True)
plt.savefig('charts/nine_box_personas.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())

**How to read this chart**

* **Three panels, one per Activity level:** High (left), Mid (centre) and Low (right). The panel title gives the number of partners at that level.
* **Inside each panel, a 3 × 3 grid:** rows are **Scale & potential** (High at the top, Low at the bottom); columns are **Profitability** (High on the left, Low on the right).
* **Each box is one of the 27 cells.** The code in the top-left corner reads Activity / Scale / Profit (for example, H/H/L = High activity, High scale, Low profit). The big number is how many partners sit in that cell.
* **The colour and name give the persona.** Cells that call for the same action share a persona, so one persona can span several cells.
* **White boxes with grey zeros are empty cells:** no partner has that combination.

**What it shows on this sample (illustration only)**

* **The core is small.** Only 36 partners sit in H/H/H, **Business Sustainers**: active, big and profitable. Another 16 are big and profitable but slipping (**Ringfence Them**, M/H/H).
* **Big doesn't mean profitable.** 103 partners are big but low on profit (**Volume Movers**: 81 in H/H/L and 22 in M/H/L), more than twice the number of Business Sustainers.
* **The middle bleeds.** 133 partners are mid-sized with thin or negative margins (**Business Bleeders**), the largest active persona.
* **Profitable, but small for us.** 60 partners buy regularly at healthy margins, but we're a small share of their business (**Potential Goldmine**). This is where the sales team's growth time should go.
* **The tail costs.** 58 partners are small and unprofitable (**Costly Tail**), candidates for a distributor or digital ordering.
* **A large group has gone quiet.** 179 partners are Low on activity (**Gone Quiet**), almost all of them in L/L/L. None of them are still big, so none fall in the High or Mid scale rows of that panel.
* **Eight cells are empty**, including big partners with mid profit (H/H/M, M/H/M). On this sample, scale tends to come with either high or low profit, rarely in between.

The counts are from the AdventureWorks sample and show how the grid reads, not benchmarks. On your own data, the cut-offs and the mix of partners across cells will differ.

In [ ]:
lenses['Persona'] = [persona_of(a, s, p) for a, s, p in zip(lenses.Activity, lenses.Scale, lenses.Profitability)]

contrib = (lenses.groupby('Persona')
                 .agg(Partners=('Persona', 'size'), Revenue=('Monetary', 'sum'))
                 .sort_values('Revenue', ascending=False))
contrib['Revenue_pct']  = contrib['Revenue']  / contrib['Revenue'].sum()  * 100
contrib['Partners_pct'] = contrib['Partners'] / contrib['Partners'].sum() * 100

fig, ax = plt.subplots(figsize=(13, 6.2))
fig.patch.set_facecolor('#f9f8f6'); ax.set_facecolor('#f9f8f6')

y = list(range(len(contrib)))[::-1]                       # largest at the top
for yi, (per, r) in zip(y, contrib.iterrows()):
    fill, ink = PERSONA_COLOURS[per]
    ax.barh(yi, r.Revenue_pct, color=fill, edgecolor=ink, linewidth=1.2, height=0.7)
    ax.text(r.Revenue_pct + 0.8, yi, f'{r.Revenue_pct:.1f}%  of revenue',
            va='center', fontsize=12, fontweight='bold', color=ink)
    ax.text(r.Revenue_pct + 0.8, yi - 0.27, f'{int(r.Partners)} partners ({r.Partners_pct:.0f}% of partners)',
            va='center', fontsize=9.5, color='#6b7178')

ax.set_yticks(y); ax.set_yticklabels(contrib.index, fontsize=12, fontweight='bold')
for label, per in zip(ax.get_yticklabels(), contrib.index):
    label.set_color(PERSONA_COLOURS[per][1])
ax.set_xlim(0, contrib.Revenue_pct.max() * 1.35)
ax.set_xlabel('Share of net revenue in the 12-month window (%)', fontsize=11, color='#4a5057')
ax.tick_params(length=0); ax.xaxis.grid(True, color='#e3e6ea'); ax.set_axisbelow(True)
for sp in ax.spines.values(): sp.set_visible(False)
ax.set_title('Business contribution by persona', fontsize=18, fontweight='bold', loc='left', pad=14)
fig.text(0.01, -0.02, f'Net revenue {lenses.Monetary.sum():,.0f} USD from {len(lenses):,} partners, '
         f'{WINDOW_START:%d %b %Y} – {WINDOW_END - pd.Timedelta(days=1):%d %b %Y}. '
         'Never Started partners have no revenue. Illustration on the AdventureWorks sample.',
         fontsize=9.5, color='#6b7178')
plt.tight_layout()
plt.savefig('charts/persona_contribution.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

contrib.round(1)

**How to read this chart**

* **One bar per persona,** sorted from the largest share of revenue to the smallest.
* **Bar length = the persona's share of net revenue** in the 12-month window (1 Dec 2012 – 29 Nov 2013). All bars add up to 100%.
* **The bold label** gives that share. **The grey line under it** gives how many partners sit in the persona, and what share of all partners that is.
* **Compare the two numbers on each bar.** When the revenue share is much bigger than the partner share, a few partners carry a lot of business. When it's much smaller, many partners bring little.
* **Never Started** partners have no revenue, so they don't appear.

**What it shows on this sample (illustration only)**

* **Revenue is concentrated.** Volume Movers alone bring 52.9% of revenue from 16% of partners. Add Business Sustainers (18.8% from 6% of partners), and two personas bring over 70% of revenue from about a fifth of partners.
* **The biggest bar isn't the best bar.** Volume Movers lead on revenue but sit in the low-profit cells. Revenue tells you who is big; the Profitability lens tells you who is valuable. That's why each persona gets its own action.
* **Business Bleeders are many and middling.** 21% of partners bring 13.7% of revenue at thin or negative margins: a lot of effort for little return.
* **Small bars, different stories.** Potential Goldmine and Costly Tail each bring under 1% of revenue from about 9% of partners. One is profitable with headroom (grow it); the other is unprofitable (serve it cheaply).
* **Gone Quiet is the largest group but brings almost nothing.** 28% of partners bring 0.1% of revenue. Win back the ones with large potential, and hand the rest to a distributor.

The shares are from the AdventureWorks sample and show how the chart reads, not benchmarks.

In [ ]:
active_partners = lenses[lenses['Frequency'] > 0]                 # 493 partners who ordered in the window
persona_order = ['Volume Movers', 'Business Sustainers', 'Business Bleeders', 'Ringfence Them',
                 'On the Fence', 'Potential Goldmine', 'Costly Tail', 'Gone Quiet']

fig, ax = plt.subplots(figsize=(14, 7.4))
fig.patch.set_facecolor('#f9f8f6'); ax.set_facecolor('#f9f8f6')

for per in persona_order:
    d = active_partners[active_partners['Persona'] == per]
    if d.empty:
        continue
    ink = PERSONA_COLOURS[per][1]
    ax.scatter(d['Monetary'], d['GrossMargin'] / 1000, s=46, color=ink, alpha=0.75,
               edgecolor='white', linewidth=0.6, label=f'{per} ({len(d)})')

ax.axhline(0, color='#15181c', linewidth=1.2)                     # break-even line
ax.set_xscale('log'); ax.set_xlim(0.8, 1.2e6)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda v, _: f'${v:,.0f}' if v < 1_000 else (f'${v/1_000:,.0f}k' if v < 1_000_000 else f'${v/1_000_000:,.0f}M')))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}k'))
ax.set_xlabel('Net revenue in the 12-month window (log scale)', fontsize=12, color='#4a5057')
ax.set_ylabel('Gross margin (USD thousands)', fontsize=12, color='#4a5057')
ax.text(1.0,  2, 'Making money', fontsize=11, fontweight='bold', color='#17663a', va='bottom')
ax.text(1.0, -2, 'Losing money', fontsize=11, fontweight='bold', color='#a3160d', va='top')

ax.grid(True, color='#e3e6ea'); ax.set_axisbelow(True); ax.tick_params(length=0)
for sp in ax.spines.values(): sp.set_visible(False)
ax.legend(loc='lower left', frameon=False, fontsize=10, ncol=2)
ax.set_title("Revenue isn't value: one dot per partner", fontsize=19, fontweight='bold', loc='left', pad=34)
ax.text(0, 1.03, 'The biggest partners by revenue (right) include both the biggest margin earners and the biggest margin losses.',
        transform=ax.transAxes, fontsize=12, color='#4a5057')
fig.text(0.01, -0.02, f'{len(active_partners):,} partners who ordered between {WINDOW_START:%d %b %Y} and '
         f'{WINDOW_END - pd.Timedelta(days=1):%d %b %Y}, coloured by persona. Illustration on the AdventureWorks sample.',
         fontsize=10, color='#6b7178')
plt.tight_layout()
plt.savefig('charts/revenue_vs_margin.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

# Supporting numbers for the note below
top20 = active_partners.nlargest(20, 'Monetary')
print(f"Top 20 partners by revenue: {(top20['GrossMargin'] < 0).sum()} lose money | "
      f"{len(set(top20.index) & set(active_partners.nlargest(20, 'GrossMargin').index))} are also in the top 20 by margin")

**How to read this chart**

* **Each dot is one partner** who ordered in the last 12 months (493 partners), coloured by persona.
* **Left to right: net revenue,** on a log scale. Each gridline is ten times the one before ($1, $10, $100 … $1M), so small and large partners both fit on one chart.
* **Up and down: gross margin** in USD thousands. The black line is break-even. Dots above it make money; dots below it lose money.
* **Read the right-hand side carefully.** That's where the big partners sit, and where revenue and value split apart.

**What it shows on this sample (illustration only)**

* **Revenue and margin don't move together.** As revenue grows, the dots fan out both upwards and downwards instead of rising in a line.
* **Big partners split into two groups.** Business Sustainers and Ringfence Them (blue and purple) earn the most margin. Volume Movers (orange) bring similar revenue but sit well below break-even.
* **The top of the revenue ranking is a mixed bag.** Of the 20 biggest partners by revenue, 13 lose money, and only 7 are also among the 20 biggest by margin.
* **Small partners hug the break-even line.** Potential Goldmine, Costly Tail and Gone Quiet sit on the left, close to zero. Their margin is small either way, so the decision there is about growth or cost to serve, not profit.
* **Business Bleeders sit in the middle, just below the line:** mid-sized revenue, steadily losing a little.

This is why RFM-B² scores profitability as its own lens. Ranking partners by revenue alone would put several loss-making partners at the top of the list for schemes and visits.

The margins are from the AdventureWorks sample, which has unusually thin margins. Read this as how the chart works, not as a benchmark.

In [ ]:
contrib['Cum_pct'] = contrib['Revenue_pct'].cumsum()

fig, ax = plt.subplots(figsize=(14, 6.8))
fig.patch.set_facecolor('#f9f8f6'); ax.set_facecolor('#f9f8f6')
x = list(range(len(contrib)))

# Bars: each persona's share of revenue, largest first
for xi, (per, r) in zip(x, contrib.iterrows()):
    fill, ink = PERSONA_COLOURS[per]
    ax.bar(xi, r.Revenue_pct, color=fill, edgecolor=ink, linewidth=1.2, width=0.7)
    y_lab = r.Revenue_pct - 8 if r.Revenue_pct > 15 else r.Revenue_pct + 1.2      # tall bars: label inside, clear of the line
    ax.text(xi, y_lab, f'{r.Revenue_pct:.1f}%', ha='center', fontsize=12, fontweight='bold', color=ink)
    ax.text(xi, -0.12, f'{int(r.Partners)} partners ({r.Partners_pct:.0f}%)', ha='center', va='top',
            fontsize=9, color='#6b7178', transform=ax.get_xaxis_transform())

ax.set_xticks(x)
ax.set_xticklabels([p.replace(' ', '\n', 1) for p in contrib.index], fontsize=11, fontweight='bold')
for label, per in zip(ax.get_xticklabels(), contrib.index):
    label.set_color(PERSONA_COLOURS[per][1])
ax.set_ylim(0, 110)
ax.set_ylabel('Share of net revenue (%)', fontsize=11, color='#4a5057')

# Line: cumulative share, on a matching second axis
ax2 = ax.twinx()
ax2.plot(x, contrib['Cum_pct'], color='#15181c', marker='o', linewidth=2.2, markersize=7)
for xi, c in zip(x, contrib['Cum_pct']):
    ax2.text(xi - 0.1, c + 2.5, f'{c:.0f}%', ha='right', fontsize=10, fontweight='bold', color='#15181c')
ax2.axhline(80, color='#c2610c', linestyle='--', linewidth=1.3)
ax2.text(len(contrib) - 0.55, 81.5, '80% of revenue', ha='right', fontsize=10, fontweight='bold', color='#c2610c')
ax2.set_ylim(0, 110)
ax2.set_ylabel('Cumulative share of revenue (%)', fontsize=11, color='#4a5057')

for a_ in (ax, ax2):
    a_.tick_params(length=0)
    for sp in a_.spines.values(): sp.set_visible(False)
ax.yaxis.grid(True, color='#e3e6ea'); ax.set_axisbelow(True)
ax.set_title('Business contribution by persona: Pareto view', fontsize=18, fontweight='bold', loc='left', pad=14)
fig.text(0.01, -0.06, "Bars: each persona's share of net revenue, largest first. Line: cumulative share. "
         f'Net revenue {lenses.Monetary.sum():,.0f} USD, {WINDOW_START:%d %b %Y} – {WINDOW_END - pd.Timedelta(days=1):%d %b %Y}. '
         'Illustration on the AdventureWorks sample.', fontsize=9.5, color='#6b7178')
plt.tight_layout()
plt.savefig('charts/persona_pareto.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

contrib[['Partners', 'Partners_pct', 'Revenue_pct', 'Cum_pct']].round(1)

**How to read this chart**

* **Bars (left axis):** each persona's share of net revenue in the 12-month window, sorted largest first. The grey line under each name gives the persona's partner count and share of partners.
* **Black line (right axis):** the cumulative share of revenue as you add personas from left to right. Each dot shows the running total.
* **Orange dashed line:** 80% of revenue. Where the black line crosses it tells you how few personas carry most of the business.
* **Read the bars and the line together:** the bars show who is big, the line shows how quickly the business concentrates, and the partner counts show how many partners that takes.

**What it shows on this sample (illustration only)**

* **The 80% line is crossed at the third persona.** Volume Movers, Business Sustainers and Business Bleeders together bring **85% of revenue from 43% of partners**.
* **Five personas make up 99% of revenue.** Adding Ringfence Them and On the Fence takes the line to 99%, from 54% of partners.
* **The long, flat tail.** Costly Tail, Potential Goldmine and Gone Quiet together add about 1% of revenue, yet they are **46% of partners**. The line barely moves after On the Fence.
* **The tail isn't one decision.** It holds three very different personas: one to grow (Potential Goldmine), one to serve cheaply (Costly Tail) and one to win back or hand over (Gone Quiet). A Pareto chart shows where the revenue is; the personas show what to do about it.
* **Concentration isn't the same as value.** The persona that opens the curve, Volume Movers, sits in the low-profit cells. The Pareto ranks partners by revenue, while the Profitability lens checks whether that revenue is worth having.

The shares are from the AdventureWorks sample and show how the chart reads, not benchmarks.

Finally taking a columnar dump of the outcome for actions to be taken and future retention

In [ ]:
partner_table = (lenses.reset_index()
    .rename(columns={'ResellerKey':          'Partner code',
                     'ResellerName':         'Partner name',
                     'Activity_score':       'Activity lens score',
                     'Profitability_score':  'Profitability lens score',
                     'Scale_score':          'Scale & potential lens score',
                     'Monetary':             'Net sales (last 12 months)',
                     'GrossMargin':          'Gross margin value',
                     'GrossMarginPct':       'Gross margin %',
                     'WalletShare':          'Wallet share'})
    [['Partner code', 'Partner name', 'Persona',
      'Activity lens score', 'Profitability lens score', 'Scale & potential lens score',
      'Net sales (last 12 months)', 'Gross margin value', 'Gross margin %', 'Wallet share']])

# Add the dealers who never ordered, as Never Started
ns = (never_started.rename(columns={'ResellerKey': 'Partner code', 'ResellerName': 'Partner name'})
                   [['Partner code', 'Partner name']]
                   .assign(Persona='Never Started'))
partner_table = pd.concat([partner_table, ns], ignore_index=True)

# No orders in the window = zero sales and margin (lens scores stay blank for Never Started)
partner_table[['Net sales (last 12 months)', 'Gross margin value']] = \
    partner_table[['Net sales (last 12 months)', 'Gross margin value']].fillna(0)

partner_table = partner_table.sort_values('Net sales (last 12 months)', ascending=False).reset_index(drop=True)
print(f'{len(partner_table):,} partners')

partner_table.head(20).style.format({
    'Activity lens score': '{:.1f}', 'Profitability lens score': '{:.1f}', 'Scale & potential lens score': '{:.1f}',
    'Net sales (last 12 months)': '{:,.0f}', 'Gross margin value': '{:,.0f}',
    'Gross margin %': '{:.1%}', 'Wallet share': '{:.1%}'}, na_rep='–')

In [ ]:
partner_table.to_csv('partner_table.csv', index=False)

### End of code